# Fine-tuning a CNN: a hands-on lab

**Goal:** by the end of this notebook **you** will have written the code that takes a CNN trained on millions of images and adapts it to *your own* small dataset (here: **cats vs. dogs**).

| # | You will... |
|---|-------------|
| 1 | read images and inspect their dimensions |
| 2 | apply augmentations to an image |
| 3 | combine augmentations with `Compose` |
| 4 | get all images to the same size |
| 5 | write a `Dataset` from scratch and build `DataLoader`s |
| 6 | load a small pretrained model and look inside it |
| 7 | freeze layers and fine-tune |

**How this notebook works**
- ✏️ **TODO cells** are yours: replace every `...` with your own code. Read the text above the cell first.
- ✅ **Check cells** run automatic tests on what you wrote. Don't move on until they show ✅.
- 💡 **Hints** are hidden: try first, open them only if you are stuck.
- 💬 **Discuss / write** prompts: answer them in the Markdown cell below, in your own words.
- Everything runs on a laptop CPU in a few minutes. The first run downloads the pretrained weights (about 14 MB).

## 0. Setup and helpers (already written, just run it)

The cell below contains **tools we give you** so you can focus on the interesting part: plotting helpers, `check(...)` for the self-tests, and `label_from_name(path)`, which reads the class from a file name (`cat.12.jpg` → 0, `dog.3.jpg` → 1). Skim through it, but you don't need to change anything.

In [2]:
import random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms.v2 as transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import warnings
warnings.filterwarnings("ignore", message=r"The transform `ToTensor\(\)` is deprecated")  # ToTensor still works, and it is simpler to teach

print("torch", torch.__version__, "| torchvision", torchvision.__version__)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

DATA_DIR = Path("tiny_data")            # tiny_data/train and tiny_data/valid, next to this notebook
CLASS_NAMES = ["cat", "dog"]            # label 0 = cat, label 1 = dog
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---------- helpers we give you ----------
def label_from_name(path):
    """'.../cat.12.jpg' -> 0 ; '.../dog.3.jpg' -> 1"""
    return 0 if Path(path).name.startswith("cat") else 1

def check(label, test, hint=""):
    """Run test() and print a green tick or a red cross with a hint. Never stops the notebook."""
    try:
        ok = bool(test())
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + label + ("" if ok else f"   ->   {hint}"))

def show_images(imgs, titles=None, ncols=5, figsize_per=2.4):
    n = len(imgs); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per * ncols, figsize_per * nrows))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for i, (ax, im) in enumerate(zip(axes, imgs)):
        ax.imshow(im)
        if titles is not None: ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

def demo(transform, image, n=5, title=""):
    """Apply a (random) PIL->PIL transform n times to the same image and show the results."""
    show_images([transform(image) for _ in range(n)], ncols=n, figsize_per=2.6)
    if title: print(title)

def denormalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Undo Normalize so a (C,H,W) tensor can be shown with plt.imshow."""
    mean = torch.tensor(mean).view(3, 1, 1); std = torch.tensor(std).view(3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

def count_params(m, only_trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not only_trainable))

def set_bn_eval(m):
    """Put every BatchNorm layer in eval mode (use the pretrained running statistics)."""
    for module in m.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader):
    """Return (loss, accuracy) of model on loader."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        n += len(yb)
    return total_loss / n, correct / n

torch 2.13.0 | torchvision 0.28.0
device: mps


---
## 1. Reading images and looking at their dimensions

Our data is two folders of `.jpg` files: `tiny_data/train` and `tiny_data/valid`. The class is not in a sub-folder name; it is in the **file name**.

### ✏️ 1.1 List the files
Use `Path.glob` to build two **sorted** lists: `train_files` (all `.jpg` in `DATA_DIR / "train"`) and `valid_files` (same for `"valid"`).

In [ ]:
train_files = ...
valid_files = ...

In [ ]:
check("train_files has 20 files", lambda: len(train_files) == 20, "did you use glob('*.jpg') on the train folder?")
check("valid_files has 10 files", lambda: len(valid_files) == 10, "did you use glob('*.jpg') on the valid folder?")
check("lists are sorted", lambda: [f.name for f in train_files] == sorted(f.name for f in train_files), "wrap glob(...) in sorted(...)")

### ✏️ 1.2 Open an image
Open the **first training file** with PIL (`Image.open`), call `.convert("RGB")` on it (this protects us from grayscale or transparent images), store it in `img`, then print its `mode` and `size`.

<details><summary>💡 Hint</summary>
<code>Image.open(path)</code> returns an image object. It has attributes <code>.mode</code> and <code>.size</code>. Evaluating <code>img</code> alone on the last line of a cell displays it in the notebook.
</details>

In [ ]:
img = ...
print("mode:", ...)
print("size:", ...)
img

In [ ]:
check("img is a PIL image", lambda: isinstance(img, Image.Image), "use Image.open(...)")
check("img is RGB", lambda: img.mode == "RGB", "add .convert('RGB')")
check("img is the first training file", lambda: img.size == Image.open(train_files[0]).size, "open train_files[0]")

### 💬 1.3 Predict, then verify
PIL, NumPy and PyTorch do **not** agree on the order of dimensions:

| Library | Shape convention |
|---------|------------------|
| PIL `img.size` | `(width, height)` |
| NumPy array | `(height, width, channels)` |
| PyTorch tensor | `(channels, height, width)` |

**Before running any code**, write your prediction. If `img.size == (300, 280)`: what is the NumPy shape? What is the tensor shape?

_Your prediction:_ ...

Now check it. Convert `img` to a NumPy array (`np.array`) called `arr`, and to a tensor called `tens` with `transforms.ToTensor()`. Print both shapes, the NumPy dtype with its min/max, and the tensor's dtype with its min/max. 

**What does `ToTensor` do to the pixel values, besides changing the shape?**

In [ ]:
arr = ...
tens = ...
print("NumPy :", ...)
print("Tensor:", ...)

In [ ]:
check("arr has shape (H, W, 3)", lambda: arr.shape == (img.size[1], img.size[0], 3), "NumPy is (height, width, channels)")
check("tens has shape (3, H, W)", lambda: tuple(tens.shape) == (3, img.size[1], img.size[0]), "PyTorch is (channels, height, width)")

_Your answer: what does `ToTensor` do to the values?_ ...

### ✏️ 1.4 Are all the images the same size?
Build a list `sizes` with the `(width, height)` of **every** image (train and valid). Then compute `n_distinct`, the number of different sizes. Don't keep all the images in memory: open, read the size, move on.

<details><summary>💡 Hint</summary>
A list comprehension over <code>train_files + valid_files</code>, and <code>set(sizes)</code> removes duplicates.
</details>

In [ ]:
all_files = train_files + valid_files
sizes = ...
n_distinct = ...
print(f"{n_distinct} distinct sizes out of {len(sizes)} images")

In [ ]:
check("sizes has 30 entries", lambda: len(sizes) == 30, "one (width, height) per image")
check("n_distinct is right", lambda: n_distinct == len(set(Image.open(f).size for f in all_files)), "count unique (w, h) pairs")

# plot (given): each dot is one image
widths = np.array([s[0] for s in sizes]); heights = np.array([s[1] for s in sizes])
is_dog = np.array([label_from_name(f) == 1 for f in all_files])
plt.figure(figsize=(5, 5))
plt.scatter(widths[~is_dog], heights[~is_dog], label="cat")
plt.scatter(widths[is_dog], heights[is_dog], label="dog", marker="x")
plt.xlabel("width (px)"); plt.ylabel("height (px)"); plt.legend(); plt.title("Image sizes in tiny_data"); plt.show()
show_images([Image.open(f).convert("RGB") for f in all_files[:10]], [f"{f.name}\n{s}" for f, s in zip(all_files, sizes)])

### 💬 1.5 Write
A neural network needs a fixed input shape. What problem do you expect with these images? (We will solve it in Section 4.)

_Your answer:_ ...

---
## 2. Augmentations

**Data augmentation** means randomly modifying training images (flip, rotate, change colours...) so the model sees a slightly different version every epoch. It is a cheap regulariser, and it matters a lot when you only have 20 training images.

Each transform in `torchvision.transforms` is a callable object: `new_img = transform(img)`. Docs: <https://pytorch.org/vision/stable/transforms.html>

### ✏️ 2.1 Create four augmentations
Fill in the dictionary `augs` with **transform objects** (not results!) for:
- a horizontal flip that happens half of the time,
- a random rotation of up to ±25 degrees,
- a random colour change (brightness, contrast and saturation, you choose the strength),
- a random zoom crop that outputs 224×224 and keeps between 40% and 100% of the image area.

<details><summary>💡 Hint</summary>
Look for <code>RandomHorizontalFlip</code>, <code>RandomRotation</code>, <code>ColorJitter</code> and <code>RandomResizedCrop</code> in the docs. The last one has a <code>scale=(low, high)</code> argument.
</details>

In [ ]:
augs = {
    "flip":     ...,
    "rotation": ...,
    "color":    ...,
    "crop":     ...,
}

In [ ]:
check("augs has 4 callables", lambda: len(augs) == 4 and all(callable(t) for t in augs.values()), "each value must be a transform object")
check("crop outputs 224x224", lambda: augs["crop"](img).size == (224, 224), "RandomResizedCrop(224, ...)")
check("rotation keeps the size", lambda: augs["rotation"](img).size == img.size, "RandomRotation(25) keeps the image size by default")

# apply each one 5 times to the same image (given)
for name, t in augs.items():
    print(name); demo(t, img)

### 💬 2.2 Which augmentations are safe?
The rule is: **the augmented image must still have the same label.**
1. Is a horizontally flipped cat still a cat? And a *vertically* flipped cat? Would you use `RandomVerticalFlip` here?
2. You are classifying handwritten digits. Is `RandomHorizontalFlip` a good idea?
3. You are classifying ripe vs. unripe bananas. Is a strong hue change safe?

_Your answers:_ ...

### ✏️ 2.3 Explore one more
Pick an augmentation **we did not use** (`RandomPerspective`, `RandomAffine`, `GaussianBlur`, `RandomGrayscale`, `RandomAutocontrast`...). Create it, call `demo(...)` on `img`, and say in one sentence whether you would keep it for cats vs. dogs.

In [ ]:
# your code here


---
## 3. Combining transforms with `Compose`

`transforms.Compose([t1, t2, t3])` returns **one** transform that applies `t1`, then `t2`, then `t3`.

Two rules:
1. **Order matters.** Some transforms work on PIL images, others on tensors. `ToTensor()` is the bridge: everything before it works on PIL, everything after it on tensors.
2. `Normalize(mean, std)` computes `(x - mean) / std` per channel and works on **tensors**, so it goes after `ToTensor`.

Pretrained ImageNet models expect inputs normalised with `IMAGENET_MEAN` and `IMAGENET_STD` (defined in the setup cell).

### ✏️ 3.1 Your first pipeline (PIL only)
Build `augment`: a `Compose` that applies, in this order, the flip, the rotation and the colour jitter **from your `augs` dictionary**. Then call `demo(augment, img, n=6)`.

In [ ]:
augment = ...
demo(augment, img, n=6, title="Same image, same pipeline, six different outputs")

In [ ]:
check("augment is a Compose", lambda: isinstance(augment, transforms.Compose), "use transforms.Compose([...])")
check("augment returns a PIL image", lambda: isinstance(augment(img), Image.Image), "no ToTensor in this one")

### ✏️ 3.2 Pipeline with tensors and normalisation
Build `pipeline`: flip → rotation → `ToTensor` → `Normalize` with the ImageNet constants. Apply it to `img` and store the result in `x`. Print the shape and the min / max / mean of `x`.

In [ ]:
pipeline = ...
x = ...
print("shape:", ..., "| min / max / mean:", ...)

In [ ]:
check("x is a float tensor (3, H, W)", lambda: torch.is_tensor(x) and x.dtype == torch.float32 and x.shape[0] == 3, "ToTensor must be in the pipeline")
check("values were normalised (some negative)", lambda: x.min() < 0, "did you add Normalize after ToTensor?")
show_images([denormalize(pipeline(img)) for _ in range(5)], ncols=5)   # given: denormalize() undoes Normalize for display

### 💬 3.3 Break it on purpose
Put `Normalize` **before** `ToTensor` in a new `Compose` and apply it to `img`. Read the error message. What does `Normalize` expect that a PIL image is not?

_Your answer:_ ...

In [ ]:
# your code here: wrap it in try/except so the notebook doesn't stop


---
## 4. Getting all images to the same size

### ✏️ 4.1 See the problem
In Section 1 you found that the images have different sizes. Convert the **first 4 training images** to tensors with `ToTensor`, print their shapes, and then try to put them into one batch with `torch.stack`. Wrap the `torch.stack` call in `try/except RuntimeError` and print the error.

In [ ]:
tensors = ...
print("shapes:", ...)
try:
    batch = ...
except RuntimeError as e:
    print("\nCould not make a batch:\n", e)

A batch is **one** tensor of shape `(B, C, H, W)`, so every image needs the same `C, H, W`. The pretrained model we will use was trained on **224 × 224** images, so that is our target.

There are three common ways to fix the size:

| Transform | What it does | Side effect |
|-----------|--------------|-------------|
| `Resize((224, 224))` | stretches / squashes to exactly 224×224 | **distorts** the aspect ratio |
| `Resize(256)` then `CenterCrop(224)` | scales the *short* side to 256, then cuts the central 224×224 | keeps proportions, **cuts off** the borders |
| `RandomResizedCrop(224)` | random crop of random size, resized to 224×224 | randomness (an augmentation, so only for training) |

### ✏️ 4.2 Compare them
Take a landscape image: `wide = Image.open(valid_files[0]).convert("RGB")`. Fill in the dictionary `variants` with the three transforms from the table (use `Compose` for the second one). The cell below shows them side by side.

In [ ]:
wide = Image.open(valid_files[0]).convert("RGB")
print("original size:", wide.size)

variants = {
    "Resize((224,224))":        ...,
    "Resize(256)+CenterCrop":   ...,
    "RandomResizedCrop(224)":   ...,
}

In [ ]:
check("all three give 224x224", lambda: all(t(wide).size == (224, 224) for t in variants.values()), "each variant must output a 224x224 image")
show_images([wide] + [t(wide) for t in variants.values()], titles=["original"] + list(variants), ncols=4, figsize_per=3.2)

### 💬 4.3 Write
1. Which variant **distorts** the animal? Which one may **cut off** part of it?
2. Which would you use for the **training** set, and which for the **validation** set? Why must the validation pipeline have **no random transforms**?

_Your answers:_ ...

### ✏️ 4.4 Your two final pipelines
Write the pipelines we will use for the rest of the lab (`IMG_SIZE = 224`):

- `train_tf`: `RandomResizedCrop` (`scale=(0.6, 1.0)`) → horizontal flip → colour jitter (brightness and contrast 0.2) → `ToTensor` → `Normalize`
- `valid_tf`: `Resize(256)` → `CenterCrop(224)` → `ToTensor` → `Normalize`. **Nothing random.**

We use `scale=(0.6, 1.0)` so each random crop keeps at least 60% of the image: the default `(0.08, 1.0)` can crop down to a tiny patch of fur or background, and then the label “cat” or “dog” would no longer be true for that crop.

In [ ]:
IMG_SIZE = 224

train_tf = ...
valid_tf = ...

In [ ]:
check("train_tf outputs (3, 224, 224)", lambda: tuple(train_tf(wide).shape) == (3, 224, 224), "check RandomResizedCrop(IMG_SIZE) and ToTensor")
check("valid_tf outputs (3, 224, 224)", lambda: tuple(valid_tf(wide).shape) == (3, 224, 224), "Resize(256) then CenterCrop(224) then ToTensor")
check("valid_tf is deterministic", lambda: torch.equal(valid_tf(wide), valid_tf(wide)), "remove every random transform from valid_tf")
check("train_tf is random", lambda: not torch.equal(train_tf(wide), train_tf(wide)), "train_tf should contain random augmentations")
check("both are normalised", lambda: train_tf(wide).min() < 0 and valid_tf(wide).min() < 0, "add Normalize(IMAGENET_MEAN, IMAGENET_STD) at the end")

---
## 5. Writing a `Dataset` from scratch

PyTorch separates *what the data is* from *how it is delivered*:

- A **`Dataset`** knows how to give you item number `i`: `dataset[i]` returns `(image_tensor, label)`.
- A **`DataLoader`** wraps a dataset and delivers shuffled mini-batches.

To write your own `Dataset` you implement three methods:

| Method | Job |
|--------|-----|
| `__init__(self, folder, transform=None)` | collect the (sorted) list of `.jpg` paths in `folder` and remember the transform. **Don't load the images yet.** |
| `__len__(self)` | return the number of images |
| `__getitem__(self, idx)` | open image `idx` (as RGB), compute its label with `label_from_name`, apply the transform **if there is one**, return `(image, label)` |

### ✏️ 5.1 Write `CatsDogsDataset`
<details><summary>💡 Hint</summary>
In <code>__init__</code> you need exactly the same <code>sorted(Path(folder).glob(...))</code> you wrote in 1.1. In <code>__getitem__</code>, remember that <code>self.transform</code> may be <code>None</code>.
</details>

In [ ]:
class CatsDogsDataset(Dataset):
    def __init__(self, folder, transform=None):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...

### ✏️ 5.2 Create the datasets
Create `train_ds` from the train folder with `train_tf`, and `valid_ds` from the valid folder with `valid_tf`.

In [ ]:
train_ds = ...
valid_ds = ...

In [ ]:
check("len(train_ds) == 20", lambda: len(train_ds) == 20, "check __init__ and __len__")
check("len(valid_ds) == 10", lambda: len(valid_ds) == 10, "check __init__ and __len__")
check("item is a (3,224,224) tensor + int label", lambda: tuple(train_ds[0][0].shape) == (3, 224, 224) and train_ds[0][1] in (0, 1), "return (image, label) from __getitem__")
check("train labels: 10 cats, 10 dogs", lambda: sorted(train_ds[i][1] for i in range(20)).count(0) == 10, "use label_from_name(path)")

### ✏️ 5.3 DataLoaders
Create `train_loader` (batch size 4, **shuffled**) and `valid_loader` (batch size 4, **not** shuffled). Why shuffle one and not the other?

_Your answer:_ ...

In [ ]:
BATCH_SIZE = 4
train_loader = ...
valid_loader = ...

In [ ]:
xb, yb = next(iter(train_loader))
check("image batch has shape (4, 3, 224, 224)", lambda: tuple(xb.shape) == (4, 3, 224, 224), "batch_size=4")
check("label batch has shape (4,)", lambda: tuple(yb.shape) == (4,), "labels should be ints")
check("train_loader shuffles, valid_loader does not",
      lambda: isinstance(train_loader.sampler, torch.utils.data.RandomSampler) and isinstance(valid_loader.sampler, torch.utils.data.SequentialSampler),
      "shuffle=True for train, shuffle=False for valid")
show_images([denormalize(x) for x in xb], titles=[CLASS_NAMES[i] for i in yb], ncols=4, figsize_per=3)

---
## 6. Pretrained models

Training a CNN from scratch needs lots of data. **Transfer learning** starts from a network already trained on a huge dataset (**ImageNet**: 1.2 million images, 1000 classes) and re-uses what it learned: edges and textures in the early layers, object parts in the middle ones, whole objects in the late ones.

We use **MobileNetV2**: only about 3.5 million parameters, small enough to fine-tune on a laptop. (ResNet-50 has 25 million, VGG-16 has 138 million.)

### ✏️ 6.1 Load it
1. `weights = MobileNet_V2_Weights.DEFAULT` (the best available pretrained weights)
2. `model = mobilenet_v2(weights=weights)`
3. put the model in **evaluation mode**
4. print the total number of parameters (use `count_params`)

In [ ]:
weights = ...
model = ...
...
print(f"total parameters: {...:,}")

In [ ]:
check("model has ~3.5M parameters", lambda: 3_400_000 < count_params(model) < 3_600_000, "load mobilenet_v2 with the DEFAULT weights")
check("model is in eval mode", lambda: not model.training, "call model.eval()")

### ✏️ 6.2 Look inside
A torchvision classifier is usually split in two: `model.features` (the **backbone**, convolutional blocks that turn an image into a feature map) and `model.classifier` (the **head**, which turns features into class scores).

Print: how many blocks are in `model.features`, the `model.classifier`, and the last block of the backbone.

In [ ]:
# your code here


### 💬 6.3 Write
1. How many outputs does the last layer have, and why that number?
2. How many does *our* problem need?

_Your answers:_ ...

### ✏️ 6.4 What does the original model think of our images?
ImageNet contains many cat and dog breeds, so the original model should say something sensible about our photos. Every pretrained weights object carries its own preprocessing recipe, `weights.transforms()`, and the list of class names, `weights.meta["categories"]`.

Write `predict_top5(image)`, which takes a PIL image and returns a list of 5 `class_name`:
1. preprocess with `weights.transforms()`
2. add a batch dimension (`unsqueeze(0)`)
3. forward pass **without gradients**, then `softmax` over the class dimension
4. `topk(5)` and look up the names

<details><summary>💡 Hint</summary>
<code>with torch.no_grad():</code> around the forward pass. The probabilities tensor has shape (1, 1000); take element <code>[0]</code> before <code>topk</code>.
</details>

In [ ]:
categories = weights.meta["categories"]
preprocess = weights.transforms()
print(preprocess)

def predict_top5(image):
    ...

In [ ]:
res = predict_top5(Image.open(train_files[0]).convert("RGB"))
check("returns 5 (name, prob) pairs", lambda: len(res) == 5 and isinstance(res[0][0], str), "return a list of 5 tuples")

for f in [train_files[0], train_files[12]]:       # one cat, one dog
    print(f.name)
    for name in predict_top5(Image.open(f).convert("RGB")):
        print(f" {name}")

---
## 7. Freezing layers and fine-tuning

Every parameter in PyTorch has a flag `requires_grad`: if `True` the optimizer updates it (**trainable**); if `False` it is **frozen**: it still takes part in the forward pass but never changes.

Fine-tuning is mostly deciding *which parameters are trainable*. Three standard strategies:

| Strategy | Trainable | Good when |
|----------|-----------|-----------|
| **head** (feature extraction) | only the new classifier | very little data, task similar to ImageNet |
| **last_blocks** (partial) | classifier + last few backbone blocks | some data, or a moderately different task |
| **full** | everything | lots of data, or a very different task |

### ✏️ 7.1 Replace the head
Load a fresh `model = mobilenet_v2(weights=weights)`, and replace its last layer `model.classifier[1]` with a new `nn.Linear` that goes from 1280 features to **2** outputs (cat, dog). Print the layer before and after.

In [ ]:
model = mobilenet_v2(weights=weights)
print("before:", model.classifier[1])
...
print("after :", model.classifier[1])

In [ ]:
check("new head has 2 outputs", lambda: model.classifier[1].out_features == 2 and model.classifier[1].in_features == 1280, "nn.Linear(1280, 2)")

### ✏️ 7.2 Freeze the backbone
Set `requires_grad = False` for **every parameter of `model.features`** (loop over `model.features.parameters()`). The new head must stay trainable. Then print the total and trainable parameter counts. (t.requires_grad = False)

**Predict first:** how many trainable parameters should be left? (Hint: a `Linear(1280, 2)` has weights *and* biases.)

_Your prediction:_ ...

In [ ]:
...
print(f"total params    : {count_params(model):,}")
print(f"trainable params: {count_params(model, only_trainable=True):,}")

In [ ]:
check("only the head is trainable (1280*2 + 2 = 2562)", lambda: count_params(model, only_trainable=True) == 2562, "freeze model.features, but not the new Linear")

### ✏️ 7.3 Prove that frozen means frozen
Take one batch `xb, yb` from `train_loader`, put the model in train mode, compute the cross-entropy loss (`loss_fn`) and call `loss.backward()`. Then print:
- `model.features[0][0].weight.grad` (a frozen weight)
- the shape of `model.classifier[1].weight.grad` (a trainable weight)

In [ ]:
model.train()
set_bn_eval(model)   # given: keep the pretrained BatchNorm statistics (explained in 7.4)
...
print("gradient of first conv (frozen)  :", ...)
print("gradient of new head (trainable) :", ...)

In [ ]:
check("frozen weights have no gradient", lambda: model.features[0][0].weight.grad is None, "did you freeze model.features?")
check("head weights have a gradient", lambda: model.classifier[1].weight.grad is not None, "did you call loss.backward()?")

### ✏️ 7.4 The training loop for one epoch
You have written this loop before. Write `train_one_epoch(model, loader, optimizer)` that returns `(average_loss, accuracy)` over the epoch. For each batch: move to `device`, `zero_grad`, forward, loss, `backward`, `step`, and accumulate the loss (weighted by batch size) and the number of correct predictions.

Two details specific to fine-tuning:
1. Start with `model.train()` and then **`set_bn_eval(model)`** (given). MobileNet has BatchNorm layers whose running statistics would get very noisy with batches of 4 images, so we keep the pretrained statistics.
2. The optimizer will only receive the trainable parameters. You build it in the next step.

<details><summary>💡 Hint</summary>
Mirror the <code>evaluate</code> function in the setup cell, but with gradients, and with the three optimizer calls.
</details>

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    set_bn_eval(model)
    ...

In [ ]:
def fit(model, optimizer, epochs=8, verbose=True):
    """Given: runs train_one_epoch + evaluate for several epochs and records the history."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer)
        vl, va = evaluate(model, valid_loader)
        for k, v in zip(history, (tl, ta, vl, va)):
            history[k].append(v)
        if verbose:
            print(f"epoch {epoch:2d} | train loss {tl:.3f} acc {ta:.0%} | val loss {vl:.3f} acc {va:.0%}")
    return history

# quick test on a throw-away copy of the model
import copy
_m = copy.deepcopy(model).to(device)
_opt = torch.optim.Adam([p for p in _m.parameters() if p.requires_grad], lr=1e-3)
_out = train_one_epoch(_m, train_loader, _opt)
check("train_one_epoch returns (loss, accuracy)", lambda: len(_out) == 2 and 0 <= _out[1] <= 1 and _out[0] > 0, "return total_loss / n, correct / n")
check("it actually updated the head", lambda: not torch.equal(_m.classifier[1].weight.cpu(), model.classifier[1].weight), "call optimizer.step()")
del _m, _opt

### ✏️ 7.5 Train the head
The backbone is frozen (7.2), so only the new head will learn.
1. Build an `Adam` optimizer that receives **only the trainable parameters** (those with `requires_grad=True`), with learning rate `1e-3`.
2. Train for 8 epochs with `fit` and store the result in `history`.

<details><summary>💡 Hint</summary>
<code>[p for p in model.parameters() if p.requires_grad]</code> is the list of trainable parameters. Then <code>history = fit(model, optimizer, epochs=8)</code>.
</details>

In [ ]:
model = model.to(device)   # given: train on the GPU if there is one
optimizer = ...
history = ...

---
## Summary: the fine-tuning recipe

1. **Look at your data** (sizes, channels, classes) first.
2. **Preprocess like the pretrained model expects**: same input size, same mean/std normalisation.
3. **Augment only the training set**; keep validation deterministic.
4. **Wrap files in a `Dataset`** (`__init__`, `__len__`, `__getitem__`) and batch them with a `DataLoader`.
5. **Load a pretrained model** and **replace the last layer** to match your number of classes.
6. **Freeze** (`requires_grad = False`) what you want to keep, and give the optimizer **only trainable parameters**.
7. Use a **smaller learning rate for pretrained layers** than for new ones, and be careful with **BatchNorm** when batches are small.
8. Start with the head only; unfreeze more only when you have the data to justify it.

**Where to go from here:** try `resnet18`, `mobilenet_v3_small` or `efficientnet_b0` (where is the head in each of them? `print(model)`); use a bigger dataset; or unfreeze gradually (head first, then the last blocks with a small learning rate).